# API + Python in Jupyter: Weather Data Visualization Demo

## Goal
Learn how to:
1. Call a real API from Python
2. Read JSON data returned by the API
3. Convert the JSON into a pandas DataFrame
4. Visualize the results with matplotlib

## What we will do
We will use the Open-Meteo API to:
- Look up a city (geocoding endpoint)
- Request hourly weather data for that location (forecast endpoint)
- Plot temperature and precipitation over time

## Why this is useful
This is the same workflow used in many machine learning and data science projects:
- fetch data
- clean/reshape data
- inspect it
- visualize it

In [4]:
# Step 1: Import the libraries we need
# - requests: make HTTP/API calls
# - pandas: table/dataframe handling
# - matplotlib: plotting/visualization

import requests
import pandas as pd
import matplotlib.pyplot as plt

# Make plots a little easier to read in notebooks
plt.rcParams["figure.figsize"] = (12, 5)

# A tiny helper function so our API calls are clean and reusable
def get_json(url, params=None):
    """
    Send a GET request to an API endpoint and return the JSON response.
    
    Why this helper exists:
    - keeps our code short
    - gives us one place to handle request errors
    """
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()  # raises an error if the request failed (4xx/5xx)
    return response.json()

## Step 2: Use a geocoding API to find latitude/longitude

Most weather APIs need coordinates (latitude/longitude), not just a city name.

So we first call a geocoding endpoint:
- input: a city name (for example, "Weatherford")
- output: matching places with coordinates

In [6]:
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"

city_query = "Weatherford Oklahoma"   # remove comma just to simplify

geocode_params = {
    "name": city_query,
    "count": 5,
    "language": "en"
}

response = requests.get(GEOCODE_URL, params=geocode_params)

print("Status code:", response.status_code)
print("Raw text:")
print(response.text[:500])  # print first 500 chars

Status code: 200
Raw text:
{"generationtime_ms":0.7737875}


In [7]:
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"

city_query = "Weatherford"

geocode_params = {
    "name": city_query,
    "count": 5,
    "language": "en"
}

geocode_data = get_json(GEOCODE_URL, geocode_params)

print("Full response keys:", geocode_data.keys())

results = geocode_data.get("results")

if results is None:
    print("No 'results' key found.")
    print("Full JSON response:")
    print(geocode_data)
else:
    print("Number of results:", len(results))
    locations_df = pd.DataFrame(results)
    display(locations_df)

Full response keys: dict_keys(['results', 'generationtime_ms'])
Number of results: 5


,id,name,latitude,longitude,elevation,feature_code,country_code,admin1_id,admin2_id,timezone,population,postcodes,country_id,country,admin1,admin2
0,4740364,Weatherford,32.75930,-97.79725,321.0,PPLA2,US,4736286,4717644,America/Chicago,28742.0,"[76085, 76086, 76087, 76088]",6252001,United States,Texas,Parker
1,4554858,Weatherford,35.52616,-98.70757,504.0,PPL,US,4544379,4534674,America/Chicago,12126.0,[73096],6252001,United States,Oklahoma,Custer
2,4229822,Weatherford,34.26732,-83.91740,360.0,PPL,US,4197000,4198821,America/New_York,NaN,NaN,6252001,United States,Georgia,Hall
3,4312844,Weatherford Knob,37.43063,-84.99968,329.0,MT,US,6254925,4287126,America/New_York,NaN,NaN,6252001,United States,Kentucky,Casey
4,4450464,Weatherford Lake Dam,34.39843,-88.24504,154.0,DAM,US,4436296,4431274,America/Chicago,NaN,NaN,6252001,United States,Mississippi,Itawamba


In [8]:
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

forecast_params = {
    "latitude": 35.5261,
    "longitude": -98.7076,
    "hourly": "temperature_2m,precipitation",
    "forecast_days": 3,
    "timezone": "auto"
}

forecast_data = get_json(FORECAST_URL, forecast_params)

print(forecast_data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
